# 数据质量检查（ticket 18）

统计全部历史数据的质量问题：光电心率、GPS 漂移、暂停、跑步机，以及回填时已经发现的问题。结论写在 `.scratch/running-tool/issues/18-data-quality.md`。

- 只读：`open_analysis_db()` 用 DuckDB 以 `READ_ONLY` 方式挂载 `running.sqlite`；第一个和最后一个单元格读库文件的修改时间，最后一个单元格断言两者相同。
- 只输出汇总数字、分位数、计数和活动 id，不输出坐标。
- 直方图用分桶计数表代替（分析包没有绘图依赖）。
- 输出里有个人数据：提交时 `nbstripout` 会清掉输出。

In [ ]:
from datetime import datetime

import polars as pl

from running.aerobic import AEROBIC_TYPES, aerobic_runs
from running.config import analysis_db_path, data_dir, load_config
from running.db import open_analysis_db

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(200)

print("数据目录:", data_dir())
db_path = analysis_db_path()
mtime_before = db_path.stat().st_mtime_ns
print("库修改时间:", datetime.fromtimestamp(mtime_before / 1e9))
config = load_config()
db = open_analysis_db()
q = db.query

## 概况

各活动类型的数量、`time_zone` 为空的数量、没有逐秒记录的数量。

In [ ]:
q("""
select activity_type, count(*) n, sum((time_zone is null)::int) tz_null,
       sum((activity_id not in (select distinct activity_id from a.records))::int) no_records
from a.activities group by 1 order by 2 desc
""")

逐秒记录读进 polars 一次，后面各节共用。`spm` = `cadence` × 2（FIT 的跑步步频是单脚）。

In [ ]:
records = q("""
select activity_id, epoch(cast(timestamp as timestamp)) t, heart_rate hr, speed,
       cadence * 2 spm, distance, lat is not null has_gps, altitude
from a.records
""").sort("activity_id", "t")
by = "activity_id"
records = records.with_columns(
    (pl.col("t") - pl.col("t").first().over(by)).alias("elapsed_s"),
    pl.col("t").diff().over(by).alias("gap_s"),
    pl.col("hr").diff().over(by).alias("dhr"),
)
acts = q("select * from a.activities")
print(records.height, "秒;", records[by].n_unique(), "个活动有逐秒记录")

## 光电心率

### 掉线：每个活动心率缺失的比例

In [ ]:
null_share = records.group_by(by).agg(pl.col("hr").is_null().mean().alias("null_share"))
print(
    null_share.select(
        pl.len().alias("activities"),
        (pl.col("null_share") > 0.05).sum().alias("gt_5pct"),
        (pl.col("null_share") > 0.5).sum().alias("gt_50pct"),
        (pl.col("null_share") == 1).sum().alias("all_null"),
    )
)
null_share.filter(pl.col("null_share") > 0.05).join(
    acts.select(by, "start_time_local", "activity_type", "distance_m"), on=by
)

### 尖峰：相邻 1 秒的心率跳变

In [ ]:
jumps = records.filter((pl.col("gap_s") == 1) & pl.col("dhr").is_not_null()).with_columns(
    pl.col("dhr").abs().alias("abs_dhr")
)
jumps.select(
    pl.len().alias("n"),
    pl.col("abs_dhr").quantile(0.5).alias("p50"),
    pl.col("abs_dhr").quantile(0.99).alias("p99"),
    pl.col("abs_dhr").quantile(0.999).alias("p99.9"),
    (pl.col("abs_dhr") >= 10).sum().alias("ge10"),
    (pl.col("abs_dhr") >= 20).sum().alias("ge20"),
    pl.col(by).filter(pl.col("abs_dhr") >= 20).n_unique().alias("activities_ge20"),
)

### 锁步频：心率与步频相差 < 5 的秒占比

只看 `spm ≥ 140`（在跑）且有心率的秒，按开头 10 分钟和其余部分分开。

In [ ]:
running_s = records.filter((pl.col("spm") >= 140) & pl.col("hr").is_not_null()).with_columns(
    ((pl.col("hr") - pl.col("spm")).abs() < 5).alias("locked"),
    (pl.col("elapsed_s") < 600).alias("first_10min"),
)
print(running_s.group_by("first_10min").agg(pl.len().alias("n"), pl.col("locked").mean()))
lock10 = (
    running_s.filter(pl.col("first_10min"))
    .group_by(by)
    .agg(pl.col("locked").mean().alias("lock_share"), pl.len().alias("n"))
)
print(
    lock10.select(
        pl.len().alias("activities"),
        (pl.col("lock_share") > 0.3).sum().alias("gt_30pct"),
        (pl.col("lock_share") > 0.5).sum().alias("gt_50pct"),
    )
)
lock10.filter(pl.col("lock_share") > 0.3).join(acts.select(by, "start_time_local"), on=by)

### 单次最大心率

In [ ]:
max_hr = records.group_by(by).agg(
    pl.col("hr").max().alias("max_hr"), (pl.col("hr") >= 190).sum().alias("secs_ge190")
)
print(
    max_hr.select(
        pl.col("max_hr").quantile(0.5).alias("p50"),
        pl.col("max_hr").quantile(0.9).alias("p90"),
        pl.col("max_hr").quantile(0.99).alias("p99"),
        pl.col("max_hr").max().alias("max"),
        (pl.col("max_hr") >= 190).sum().alias("ge190"),
    )
)
max_hr.filter(pl.col("max_hr") >= 190).join(
    acts.select(by, "start_time_local", "activity_type", "distance_m"), on=by
).sort("max_hr", descending=True)

## GPS 漂移

逐秒累计距离与活动距离的比值；平均速度（距离 ÷ 移动时间）和最大速度的分布；逐秒速度 > 7 m/s（快于 2:23 /km）的秒。

In [ ]:
ratio = (
    records.group_by(by)
    .agg(pl.col("distance").max().alias("rec_dist"))
    .join(acts.filter(pl.col("distance_m") > 0).select(by, "distance_m"), on=by)
    .with_columns((pl.col("rec_dist") / pl.col("distance_m")).alias("ratio"))
)
print(
    ratio.select(
        pl.len().alias("activities_with_distance"),
        pl.col("ratio").quantile(0.01).alias("p1"),
        pl.col("ratio").median().alias("p50"),
        pl.col("ratio").quantile(0.99).alias("p99"),
        ((pl.col("ratio") - 1).abs() > 0.05).sum().alias("off_gt_5pct"),
    )
)
avg_speed = pl.col("distance_m") / pl.col("moving_duration_s")
acts.group_by("activity_type").agg(
    pl.len().alias("n"),
    pl.col("max_speed_mps").median().alias("max_speed_p50"),
    pl.col("max_speed_mps").quantile(0.99).alias("max_speed_p99"),
    (pl.col("max_speed_mps") > 7).sum().alias("max_speed_gt7"),
    avg_speed.quantile(0.01).alias("avg_speed_p1"),
    avg_speed.median().alias("avg_speed_p50"),
    avg_speed.quantile(0.99).alias("avg_speed_p99"),
).sort("n", descending=True)

In [ ]:
fast = (
    records.group_by(by)
    .agg(
        (pl.col("speed") > 7).sum().alias("secs_gt7"),
        (pl.col("speed") > 8).sum().alias("secs_gt8"),
        pl.col("speed").filter(pl.col("speed") > 7).sum().alias("dist_in_gt7_m"),
        pl.col("speed").max().alias("max_speed"),
    )
    .filter(pl.col("secs_gt7") > 0)
)
print(fast.select(pl.len().alias("activities"), pl.col("secs_gt7").sum(), pl.col("secs_gt8").sum()))
fast.join(acts.select(by, "start_time_local", "activity_type", "distance_m"), on=by).sort(
    "secs_gt7", descending=True
)

## 暂停：相邻样本的时间差

分桶计数（直方图），按年份拆开看智能记录的影响，以及长间隔前后的速度（间隔后速度为 0 说明是暂停后恢复）。

In [ ]:
gaps = records.filter(pl.col("gap_s").is_not_null())
gaps.with_columns(
    pl.col("gap_s")
    .bin_intervals(
        [1, 2, 5, 10, 30, 60, 300],
        labels=["<=1", "1-2", "2-5", "5-10", "10-30", "30-60", "60-300", ">300"],
        right_closed=True,
    )
    .alias("gap_bucket")
).group_by("gap_bucket").len().sort("gap_bucket")

In [ ]:
gap_2_10 = pl.col("gap_s").is_between(2, 10, closed="right")
gaps.join(acts.select(by, "start_time_local"), on=by).with_columns(
    pl.col("start_time_local").str.slice(0, 4).alias("year")
).group_by("year").agg(
    pl.col(by).n_unique().alias("activities"),
    gap_2_10.sum().alias("gap_2_10"),
    pl.col(by).filter(gap_2_10).n_unique().alias("activities_gap_2_10"),
    pl.col("gap_s").is_between(10, 30, closed="right").sum().alias("gap_10_30"),
    (pl.col("gap_s") > 30).sum().alias("gap_gt30"),
).sort("year")

In [ ]:
print("pause_gap_s =", config.run_walk.pause_gap_s)
print("max_sample_gap_s =", config.load.max_sample_gap_s)
records.with_columns(pl.col("speed").shift(1).over(by).alias("speed_before")).filter(
    pl.col("gap_s") > 10
).with_columns(
    pl.col("gap_s")
    .bin_intervals([30, 60, 300], labels=["10-30", "30-60", "60-300", ">300"], right_closed=True)
    .alias("gap_bucket")
).group_by("gap_bucket").agg(
    pl.len().alias("n"),
    pl.col("speed_before").median().alias("speed_before_p50"),
    pl.col("speed").median().alias("speed_after_p50"),
).sort("gap_bucket")

## 跑步机

没有 `treadmill_running` 类型的活动。跑步类型里既没有 GPS 也没有海拔的活动可能是跑步机；看它们有没有进有氧能力（`AEROBIC_TYPES` 只按类型过滤）。

In [ ]:
no_gps = (
    records.group_by(by)
    .agg(
        pl.col("has_gps").any().alias("gps"),
        pl.col("altitude").count().alias("altitude_n"),
        pl.col("spm").median().alias("spm_p50"),
    )
    .filter(~pl.col("gps"))
    .join(
        acts.select(by, "start_time_local", "activity_type", "distance_m", "moving_duration_s"),
        on=by,
    )
    .filter(pl.col("activity_type").str.contains("run"))
)
print("AEROBIC_TYPES:", sorted(AEROBIC_TYPES))
no_gps

In [ ]:
road = aerobic_runs(db, config)
print(road.height, "次路跑;", road["valid"].sum(), "次有效")
road.filter(pl.col(by).is_in(no_gps[by].implode())).select(
    by, "start_time_local", "steady_s", "valid", "speed_mps"
)

GPS 漂移对有氧能力的影响：上面速度 > 7 m/s 的活动里哪些被判为有效。

In [ ]:
road.filter(pl.col(by).is_in(fast[by].implode()) & pl.col("valid")).select(
    by, "start_time_local", "steady_s", (1000 / pl.col("speed_mps")).alias("pace_s_per_km")
)

## 已经发现的问题

### 没有逐秒记录的活动

In [ ]:
acts.filter(~pl.col(by).is_in(records[by].unique().implode())).select(
    by,
    "start_time_local",
    "start_time_utc",
    "activity_type",
    "distance_m",
    "duration_s",
    "avg_hr_bpm",
)

### `time_zone`：本地时间减 UTC 的偏移

In [ ]:
local = pl.col("start_time_local").str.to_datetime()
utc = pl.col("start_time_utc").str.replace("Z$", "").str.to_datetime()
acts.with_columns(((local - utc).dt.total_minutes() / 60).alias("offset_h")).group_by(
    "offset_h", "time_zone"
).len().sort("len", descending=True)

### 每日静息心率和最低心率

In [ ]:
q("""
select count(*) n_days,
       count(resting_hr_bpm) resting_n,
       quantile_cont(resting_hr_bpm, [0.1, 0.5, 0.9]) resting_q,
       sum((resting_hr_bpm > min_hr_bpm)::int) resting_gt_min,
       count(min_hr_bpm) min_n,
       quantile_cont(min_hr_bpm, [0.1, 0.5, 0.9]) min_q
from a.daily_wellness
""")

## 有氧能力的离散程度

有效路跑的配速与前后 15 次滚动中位数的偏离。偏离不一定是坏数据：真实配速变化和温度都会造成偏离（温度见 19）。

In [ ]:
valid = (
    road.filter(pl.col("valid"))
    .sort("start_time_local")
    .with_columns((1000 / pl.col("speed_mps")).alias("pace"))
)
valid = valid.with_columns(
    pl.col("pace").rolling_median(15, center=True, min_samples=5).alias("pace_rolling")
).with_columns(((pl.col("pace") - pl.col("pace_rolling")) / pl.col("pace_rolling")).alias("dev"))
print("有效路跑配速中位数 (s/km):", valid["pace"].median())
valid.select(
    pl.len().alias("n"),
    (pl.col("dev").abs() > 0.08).sum().alias("off_gt_8pct"),
    pl.col("dev").min().alias("min_dev"),
    pl.col("dev").max().alias("max_dev"),
)

In [ ]:
db.close()
mtime_after = db_path.stat().st_mtime_ns
print("库修改时间:", datetime.fromtimestamp(mtime_after / 1e9))
assert mtime_after == mtime_before, "running.sqlite 被改动了"